# Unlimited-OCR demo (Colab T4 / Kaggle T4×2)

Runs `baidu/Unlimited-OCR` via the Transformers path, serves a Gradio web UI, and converts detected tables to Excel.

**Before running:**
- **Colab:** Runtime → Change runtime type → **T4 GPU**
- **Kaggle:** Settings → Accelerator → **GPU T4 ×2**, and Settings → **Internet ON** (needs phone-verified account)

This is a *demo*: the public `gradio.live` link lives only while the notebook runs.

## 1. Environment check (which platform + GPU)

In [1]:
import os

if os.path.exists('/kaggle'):
    PLATFORM = 'kaggle'
elif 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    PLATFORM = 'colab'
else:
    PLATFORM = 'other'
print('Platform:', PLATFORM)

!nvidia-smi

Platform: kaggle
Thu Jun 25 09:21:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

## 2. Install dependencies
torch / torchvision already ship with CUDA on both platforms — do not reinstall them.
`pandas` + `openpyxl` are for the tables→Excel step.

In [2]:
!pip install -q "transformers==4.57.1" einops addict easydict pymupdf matplotlib Pillow gradio pandas openpyxl lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.5 MB/s eta 0:00:00


## 3. Load the model (once)
Tries bf16 first (model-card spec), falls back to fp16 if the Turing GPU rejects a kernel.
Pinned to `cuda:0` — a second T4 on Kaggle stays idle (the model fits on one).

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = 'baidu/Unlimited-OCR'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = None
for dtype in (torch.bfloat16, torch.float16):
    try:
        model = AutoModel.from_pretrained(
            model_name, trust_remote_code=True,
            use_safetensors=True, torch_dtype=dtype,
        ).eval().cuda()          # -> cuda:0
        print('Loaded in', dtype)
        break
    except Exception as e:
        print(f'{dtype} load failed:', e)

if model is None:
    raise RuntimeError('Model failed to load in both bf16 and fp16 — see tracebacks above.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- deepencoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- modeling_unlimitedocr.py
- conversation.py
- configuration_deepseek_v2.py
- modeling_deepseekv2.py
- deepencoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded in torch.bfloat16


## 4. Inference helper that actually returns the text
`model.infer()` **prints** its result to stdout and doesn't reliably return it — that's why
output showed in the Colab console but not in Gradio. This wrapper captures stdout (and reads the
saved result file as a fallback) so the text is returned to the caller.

In [17]:
import io, contextlib, glob, os, tempfile, fitz
os.makedirs('out', exist_ok=True)

def _read_saved(output_path):
    # Grab the most recent markdown/text file the model wrote, if any
    cands = []
    for ext in ('*.mmd', '*.md', '*.txt'):
        cands += glob.glob(os.path.join(output_path, '**', ext), recursive=True)
    if not cands:
        return ''
    latest = max(cands, key=os.path.getmtime)
    with open(latest, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def run_infer(multi=False, **kwargs):
    """Call model.infer / infer_multi and return the parsed text."""
    kwargs.setdefault('output_path', 'out')
    kwargs['save_results'] = True

    # Ensure the tokenizer has a pad token set globally instead of passing it to infer
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    buf = io.StringIO()
    fn = model.infer_multi if multi else model.infer

    with contextlib.redirect_stdout(buf):
        # We remove pad_token_id from kwargs because the model's .infer() wrapper doesn't accept it
        ret = fn(tokenizer, **kwargs)

    captured = buf.getvalue().strip()
    if isinstance(ret, str) and ret.strip():
        return ret.strip()
    if captured:
        return captured
    return _read_saved(kwargs['output_path']) or '(no text returned)'

def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path); mat = fitz.Matrix(dpi / 72, dpi / 72)
    tmp = tempfile.mkdtemp(); paths = []
    for i, page in enumerate(doc):
        p = os.path.join(tmp, f'page_{i+1:04d}.png')
        page.get_pixmap(matrix=mat).save(p); paths.append(p)
    doc.close(); return paths

## 5. Quick single-image test
Upload an image to the file panel, then set its path below.

In [18]:
text = run_infer(
    prompt='<image>document parsing.',
    image_file='image2.jpeg',          # <-- change to your uploaded file
    base_size=1024, image_size=640, crop_mode=True,
    max_length=8192, no_repeat_ngram_size=35, ngram_window=128,
)
print(text[:2000])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
image: 0it [00:00, ?it/s]
other: 100%|██████████| 3/3 [00:00<00:00, 32939.56it/s]

<|det|>text [102, 0, 430, 95]<|/det|>DEAR DAD
<|det|>text [62, 128, 959, 725]<|/det|>School is really great. I am making left of friend and studying very hard.
WITH ALL MY #H#I, I simply can't THINK of anything I need, if you want, you can just #ENO ME A CHAD, at I would love to hear from you.
<|det|>text [493, 765, 711, 999]<|/det|>Love,
You Dan
===============save results:===============


## 6. Tables → Excel
The model emits tables as markdown pipe-tables (and sometimes HTML). This extracts every table
and writes one sheet per table to an `.xlsx`. Use it for papers, statements, price lists, etc.

In [19]:
import re, pandas as pd

def _parse_pipe_table(block):
    rows = []
    for ln in block:
        cells = [c.strip() for c in ln.strip().strip('|').split('|')]
        rows.append(cells)
    # drop the |---|---| separator row(s)
    rows = [r for r in rows if not all(set(c) <= set('-: ') and c != '' for c in r)]
    if len(rows) < 2:
        return None
    header, *body = rows
    width = len(header)
    body = [(r + [''] * width)[:width] for r in body]  # pad/trim ragged rows
    return pd.DataFrame(body, columns=header)

def extract_tables(md_text):
    tables = []
    # 1) HTML tables (pandas reads these directly)
    if '<table' in md_text.lower():
        try:
            tables += pd.read_html(md_text)
        except Exception:
            pass
    # 2) markdown pipe-tables
    block = []
    for ln in md_text.splitlines():
        if ln.strip().startswith('|'):
            block.append(ln)
        else:
            if len(block) >= 2:
                df = _parse_pipe_table(block)
                if df is not None:
                    tables.append(df)
            block = []
    if len(block) >= 2:
        df = _parse_pipe_table(block)
        if df is not None:
            tables.append(df)
    return tables

def tables_to_excel(md_text, xlsx_path='out/tables.xlsx'):
    tables = extract_tables(md_text)
    if not tables:
        return None, 0
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as xw:
        for i, df in enumerate(tables):
            df.to_excel(xw, sheet_name=f'Table{i+1}'[:31], index=False)
    return xlsx_path, len(tables)

# Try it on the last result:
path, n = tables_to_excel(text)
print(f'Wrote {n} table(s) to {path}' if path else 'No tables found in this document.')

No tables found in this document.


## 7. Gradio web app (public link)
Shows rendered output, raw markdown, and a downloadable Excel of any tables found.

In [22]:
import gradio as gr

def handle_image(image_path):
    if not image_path:
        return 'Please upload an image.', '', None
    text = run_infer(
        prompt='<image>document parsing.', image_file=image_path,
        base_size=1024, image_size=640, crop_mode=True,
        max_length=32000, no_repeat_ngram_size=35, ngram_window=128,
    )
    xlsx, n = tables_to_excel(text, 'out/image_tables.xlsx')
    return text, text, xlsx

def handle_pdf(pdf_path):
    if not pdf_path:
        return 'Please upload a PDF.', '', None
    pages = pdf_to_images(pdf_path)
    text = run_infer(
        multi=True, prompt='<image>Multi page parsing.', image_files=pages,
        image_size=1024, max_length=32000, no_repeat_ngram_size=35, ngram_window=1024,
    )
    xlsx, n = tables_to_excel(text, 'out/pdf_tables.xlsx')
    return text, text, xlsx

with gr.Blocks() as demo:
    gr.Markdown('## Unlimited-OCR demo — parse documents, export tables to Excel')
    with gr.Tab('Image'):
        img = gr.Image(type='filepath', label='Upload document image')
        b1 = gr.Button('Parse', variant='primary')
        with gr.Tabs():
            with gr.Tab('Rendered'):
                r1 = gr.Markdown()
            with gr.Tab('Raw markdown'):
                raw1 = gr.Code(language='markdown')
        xl1 = gr.File(label='Tables as Excel (if any)')
        b1.click(handle_image, img, [r1, raw1, xl1])
    with gr.Tab('PDF'):
        pdf = gr.File(type='filepath', label='Upload PDF', file_types=['.pdf'])
        b2 = gr.Button('Parse', variant='primary')
        with gr.Tabs():
            with gr.Tab('Rendered'):
                r2 = gr.Markdown()
            with gr.Tab('Raw markdown'):
                raw2 = gr.Code(language='markdown')
        xl2 = gr.File(label='Tables as Excel (if any)')
        b2.click(handle_pdf, pdf, [r2, raw2, xl2])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f13f788b0cbac7daa6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://f13f788b0cbac7daa6.gradio.live


## 8. Stage 2 — turn OCR text into structured fields (receipts, forms, invoices)

Tables → Excel works because markdown already encodes the grid. **Receipts and forms are different**:
you want labeled fields (`vendor`, `date`, `total`), not a table. That's a second step — feed the OCR
text to an LLM and ask for structured JSON. This uses the Claude API (`claude-opus-4-8`).

**You need an Anthropic API key** for this section. On Colab, add it as a secret named `ANTHROPIC_API_KEY`
(left sidebar → 🔑 → add, then toggle notebook access) or you'll be prompted to paste it.
Get one at https://console.anthropic.com . Receipts are tiny inputs, so cost is fractions of a cent each.

In [46]:
!pip install -q -U google-generativeai
import os
import google.generativeai as genai

# Force stable API and clear any cached beta routing
os.environ['GOOGLE_API_USE_MTLS_ENDPOINT'] = 'never'

api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY").strip()
    print("Gemini API key loaded from Colab Secrets.")
except Exception:
    import getpass
    api_key = getpass.getpass("Enter Google API Key (from AI Studio): ").strip()

if not api_key or len(api_key) < 10:
    print("Warning: API Key looks invalid or empty.")
else:
    # Use the 'rest' transport
    genai.configure(api_key=api_key, transport='rest')
    # Switching to Gemini 2.5 Flash
    ai = genai.GenerativeModel("models/gemini-2.5-flash")
    print("Gemini client ready (Gemini 2.5 Flash).")

Enter Google API Key (from AI Studio): ··········
Gemini client ready (Gemini 2.5 Flash).


In [47]:
import json
from pydantic import BaseModel
from typing import List, Optional

# --- A typed schema for receipts ---
class LineItem(BaseModel):
    description: str
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    amount: Optional[float] = None

class Receipt(BaseModel):
    vendor: Optional[str] = None
    date: Optional[str] = None
    currency: Optional[str] = None
    items: List[LineItem]
    subtotal: Optional[float] = None
    tax: Optional[float] = None
    total: Optional[float] = None

# --- A flexible schema for any form / invoice ---
class FormField(BaseModel):
    field: str
    value: str

class FormExtraction(BaseModel):
    document_type: Optional[str] = None
    fields: List[FormField]

def extract_structured_data(prompt, schema_class):
    """Helper to get structured JSON from Gemini using Pydantic schemas."""
    schema_json = json.dumps(schema_class.model_json_schema())
    full_prompt = f"{prompt}\n\nReturn the data as a single JSON object strictly matching this schema: {schema_json}"

    try:
        # Ensure we use the stable v1 endpoint and the standard model name
        response = ai.generate_content(
            full_prompt,
            generation_config=genai.types.GenerationConfig(
                response_mime_type="application/json",
            )
        )
        if not response.text:
            return None
        return schema_class.model_validate_json(response.text)
    except Exception as e:
        print(f"Extraction Error: {e}")
        return None

def extract_receipt(ocr_text):
    prompt = (
        "Extract the receipt fields from this OCR text. Use null for anything not "
        "present; never invent values.\n\n" + ocr_text
    )
    return extract_structured_data(prompt, Receipt)

def extract_fields(ocr_text, instruction='Extract every labeled field as field/value pairs.'):
    prompt = instruction + "\n\n" + ocr_text
    return extract_structured_data(prompt, FormExtraction)

# Demo on the OCR result from section 5:
if 'text' in globals():
    print("Processing extraction...")
    receipt = extract_receipt(text)
    if isinstance(receipt, Receipt):
        print("Successfully extracted JSON:")
        print(receipt.model_dump_json(indent=2))
    else:
        print("Failed to extract structured data. Check if the 'ai' client was correctly initialized in cell 6bb40783.")
else:
    print("Variable 'text' not found. Please run the OCR test in Section 5 first.")

Processing extraction...
Successfully extracted JSON:
{
  "vendor": null,
  "date": null,
  "currency": null,
  "items": [],
  "subtotal": null,
  "tax": null,
  "total": null
}


In [ ]:
# Test with actual receipt-like text to verify the schema extraction
simulated_receipt_text = """
STARBUCKS STORE #12345
123 MAIN STREET, SEATTLE, WA

06/25/2024 10:15 AM

LATTE            $5.50
CROISSANT        $3.75

SUBTOTAL         $9.25
TAX (8%)         $0.74
TOTAL            $9.99
"""

print("Testing Gemini extraction with a simulated receipt...")
receipt_data = extract_receipt(simulated_receipt_text)

if receipt_data:
    display(receipt_data.model_dump())
else:
    print("Extraction failed.")

## 9. Full web app — OCR + tables→Excel + receipts/forms→fields

This is the complete app: **Image** and **PDF** tabs (parse + table export, no API key needed) plus
**Receipt** and **Form** tabs that run Stage 2 (needs the Claude client from section 8).

Run sections 1–8 first, then run this cell. It supersedes section 7 — use this one as your final app.
(If you don't want the Claude dependency, just run section 7 instead; the Receipt/Form tabs here will
tell you to run section 8.)

In [ ]:
import gradio as gr

def ocr_image_text(image_path):
    return run_infer(
        prompt='<image>document parsing.', image_file=image_path,
        base_size=1024, image_size=640, crop_mode=True,
        max_length=8192, no_repeat_ngram_size=35, ngram_window=128,
    )

def receipt_to_excel(receipt, path='out/receipt.xlsx'):
    items = pd.DataFrame([i.model_dump() for i in receipt.items]) if receipt.items else pd.DataFrame()
    summary = pd.DataFrame([{k: v for k, v in receipt.model_dump().items() if k != 'items'}])
    with pd.ExcelWriter(path, engine='openpyxl') as xw:
        summary.to_excel(xw, sheet_name='Summary', index=False)
        if not items.empty:
            items.to_excel(xw, sheet_name='Items', index=False)
    return path

# ---- tab handlers ----
def app_image(image_path):
    if not image_path:
        return 'Please upload an image.', '', None
    t = ocr_image_text(image_path)
    xlsx, _ = tables_to_excel(t, 'out/image_tables.xlsx')
    return t, t, xlsx

def app_pdf(pdf_path):
    if not pdf_path:
        return 'Please upload a PDF.', '', None
    pages = pdf_to_images(pdf_path)
    t = run_infer(multi=True, prompt='<image>Multi page parsing.', image_files=pages,
                  image_size=1024, max_length=8192, no_repeat_ngram_size=35, ngram_window=1024)
    xlsx, _ = tables_to_excel(t, 'out/pdf_tables.xlsx')
    return t, t, xlsx

def app_receipt(image_path):
    if not image_path:
        return 'Please upload a receipt image.', None
    if 'ai' not in globals():
        return 'Run section 8 first (Claude client not set up).', None
    receipt = extract_receipt(ocr_image_text(image_path))
    return receipt.model_dump_json(indent=2), receipt_to_excel(receipt)

def app_form(image_path, instruction):
    if not image_path:
        return 'Please upload a form/invoice image.', None
    if 'ai' not in globals():
        return 'Run section 8 first (Claude client not set up).', None
    extracted = extract_fields(ocr_image_text(image_path),
                               instruction or 'Extract every labeled field as field/value pairs.')
    df = pd.DataFrame([f.model_dump() for f in extracted.fields])
    path = 'out/form_fields.xlsx'
    df.to_excel(path, index=False)
    return extracted.model_dump_json(indent=2), path

with gr.Blocks() as app:
    gr.Markdown('## Unlimited-OCR — document parsing, tables to Excel, receipts & forms to fields')

    with gr.Tab('Image'):
        img = gr.Image(type='filepath', label='Upload document image')
        gr.Button('Parse', variant='primary').click(
            app_image, img,
            [gr.Markdown(label='Rendered'), gr.Code(label='Raw markdown'),
             gr.File(label='Tables as Excel')])

    with gr.Tab('PDF'):
        pdf = gr.File(type='filepath', label='Upload PDF', file_types=['.pdf'])
        gr.Button('Parse', variant='primary').click(
            app_pdf, pdf,
            [gr.Markdown(label='Rendered'), gr.Code(label='Raw markdown'),
             gr.File(label='Tables as Excel')])

    with gr.Tab('Receipt'):
        rimg = gr.Image(type='filepath', label='Upload receipt image')
        gr.Button('Extract fields', variant='primary').click(
            app_receipt, rimg,
            [gr.Code(label='Structured JSON', language='json'),
             gr.File(label='Receipt as Excel')])

    with gr.Tab('Form / Invoice'):
        fimg = gr.Image(type='filepath', label='Upload form or invoice image')
        finstr = gr.Textbox(label='What to extract (optional)',
                            placeholder='e.g. Extract invoice number, due date, and total')
        gr.Button('Extract fields', variant='primary').click(
            app_form, [fimg, finstr],
            [gr.Code(label='Structured JSON', language='json'),
             gr.File(label='Fields as Excel')])

# Added .queue() and set prevent_thread_lock to help with asyncio issues in notebooks
app.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea84e8779f5a273b29.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error